In [ ]:

import pandas as pd
import yaml

    
try:
    with open("../cfg.yaml", "r") as file:
        cfg = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

In [ ]:
#importing the raw data of energy and co2
raw_df=pd.read_csv(cfg['input_data']['file1'])
df=raw_df.copy()
df.head(1)

In [ ]:
##to know which data is useful for the project, we read the description of the dataset given per column:
codebook_df=pd.read_csv(cfg['input_data']['file2'])
code_df=codebook_df.copy()
code_df.head(1)

In [ ]:
#we take just the columns we definetly will need:
cols=['country','year','iso_code','population','co2','co2_including_luc','co2_growth_prct',
      'co2_including_luc_growth_prct','co2_per_unit_energy','co2_including_luc_per_unit_energy',
     'primary_energy_consumption']

df=df[cols]
df.head(2)

In [ ]:
code_df[(code_df['column'].isin(cols))][['column','description','unit']]

In [ ]:
#Now we only take the data for the timewindow we defined (2014-2024)
df.country.unique()
df=df[(df.year>=2014) & (df.year<=2024)]
df.head(2)

In [ ]:
df = df.rename(columns={
    'population':'pop',
    'co2_including_luc': 'co2_luc',
    'co2_growth_prct':'co2_grow_prct',
    'co2_including_luc_growth_prct':'co2_luc_prct',
    'co2_per_unit_energy':'co2_p_ener',
    'co2_including_luc_per_unit_energy':'co2_luc_p_ener',
    'primary_energy_consumption':'prim_ener_cons'
})
df.info()

In [ ]:
#where iso_code is null
nulls=df[df.iso_code.isnull()==True]
display(nulls.country.unique())


In [ ]:
#here we see that nulls are all region groups (continents) and Kosovo is the only country
#then we drop all, except Kosovo as this is not a region but a country.
df = df[(df['iso_code'].notnull()) | (df['country'] == 'Kosovo')]
df.info()

In [ ]:
#the only country without iso code.
df[df['iso_code'].isnull()].country.unique()

In [ ]:
#importing the csv region-mapping from OWD
region_df=pd.read_csv(cfg['input_data']['file3'])
reg_df=region_df.copy()
reg_df.head(1)

In [ ]:
reg_df = reg_df.rename(columns={'Entity':'country','World region according to OWID': 'Region'})
reg_df=reg_df.drop(columns=['Code','Year'])
reg_df

In [ ]:
#now we can merge the mapping and our data set.
df = df.merge(reg_df[['country', 'Region']], on='country', how='left')

In [ ]:
df=df.rename(columns={'Region':'region'})
df.head(1)

In [ ]:
df[df.region.isnull()==True].country.unique()

In [ ]:
df = df[df['country'] != 'Antarctica']
df.info()

In [ ]:
print(f"Country without iso code: {df[df.iso_code.isnull()==True].country.unique()}")

In [ ]:
df[df.co2.isnull()==True].country.unique()

In [ ]:
df= df[-df['country'].isin(['Monaco', 'San Marino', 'Vatican'])]
df.info()

In [ ]:
#Now the only missing values are for prim_ener_cons, those are for the countries:
df[df.prim_ener_cons.isnull()==True].country.unique()

In [ ]:
# Save df to the output path defined in config, with reset index
df.to_csv(cfg['output_data']['file1'], index=False)

print(f"File saved to: {cfg['output_data']['file1']}")